In [1]:
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity

EARTH_RADIUS_KM = 6371

restaurants = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/processed/restaurants_clean.csv")

taxonomy = pd.read_csv("/Users/edgardomosesezekielaverilla/Restaurant-Recommendation-System-Redux/data/reference/category_mapping.csv", encoding="cp1252")

In [2]:
"""Constants"""

CUISINE_WEIGHT = 3.0
TYPE_WEIGHT = 2.0
EXPERIENCE_WEIGHT = 1.5
NUMERIC_WEIGHT = 1.0

SIMILARITY_WEIGHT = 0.70
PROXIMITY_WEIGHT = 0.30
DECAY_FACTOR = 10
SEARCH_RADIUS_KM = 20

In [3]:
selected_restaurant = restaurants[
    (restaurants["name"] == "Ichiban") &
    (restaurants["city"] == "Nashville") &
    (restaurants["state"] == "TN")
].iloc[0]

In [4]:
from sklearn.metrics.pairwise import haversine_distances
import numpy as np

def filter_by_radius(
    restaurants,
    selected_restaurant,
    radius_km=15
):
    """
    Return restaurants within a given radius of the
    selected restaurant.
    """

    selected_coords = np.radians([
        [
            selected_restaurant["latitude"],
            selected_restaurant["longitude"]
        ]
    ])

    restaurant_coords = np.radians(
        restaurants[
            ["latitude", "longitude"]
        ]
    )

    distances = haversine_distances(
        selected_coords,
        restaurant_coords
    )[0]

    distances_km = distances * EARTH_RADIUS_KM

    filtered_restaurants = restaurants.copy()

    filtered_restaurants["distance_km"] = distances_km

    filtered_restaurants = filtered_restaurants[
        filtered_restaurants["distance_km"] <= radius_km
    ]

    return filtered_restaurants.reset_index(drop=True)


In [5]:
subset_restaurants = filter_by_radius(
    restaurants=restaurants,
    selected_restaurant=selected_restaurant,
    radius_km=10
)

subset_restaurants = subset_restaurants.sort_values(
    by="distance_km"
).reset_index(drop=True)

subset_restaurants.head(3)

,business_id,name,city,state,postal_code,latitude,longitude,stars,review_count,categories,attributes,hours,is_open,distance_km
0,WrgdQF8kzvONbZctSPlF4A,Ichiban,Nashville,TN,37201,36.162124,-86.775627,4.0,144,"Sushi Bars, Japanese, Restaurants","{'NoiseLevel': ""u'quiet'"", 'OutdoorSeating': '...","{'Monday': '17:0-22:0', 'Tuesday': '17:0-22:0'...",0,0.000000
1,Dp6QT6_evsKemZNKgtJ7nA,REPUBLIC quality food & drink,Nashville,TN,37201,36.162162,-86.775591,4.0,16,"Gastropubs, Restaurants","{'WheelchairAccessible': 'True', 'RestaurantsR...","{'Monday': '16:0-0:0', 'Tuesday': '16:0-0:0', ...",1,0.005356
2,qnOffBt8iV6umLvDZ5PNDw,Music City Chicken,Nashville,TN,37201,36.162162,-86.775591,4.5,132,"Restaurants, Food, Sandwiches, Beer, Wine & Sp...","{'RestaurantsReservations': 'False', 'OutdoorS...","{'Monday': '11:0-22:0', 'Tuesday': '11:0-22:0'...",0,0.005356


In [6]:
restaurants = subset_restaurants.copy()

In [7]:
print(len(restaurants))

2034


In [8]:
def calculate_proximity_score(distance_km, decay_factor=10):
    distance_km = np.maximum(distance_km, 0)
    return np.exp(-distance_km / decay_factor)

In [9]:
restaurants["proximity_score"] = calculate_proximity_score(
    restaurants["distance_km"],
    decay_factor=DECAY_FACTOR
)

In [10]:
restaurants[
    ["name", "city", "distance_km"]
].head(10)

,name,city,distance_km
0,Ichiban,Nashville,0.000000
1,REPUBLIC quality food & drink,Nashville,0.005356
2,Music City Chicken,Nashville,0.005356
3,The Stillery,Nashville,0.015862
4,Piranha's Bar & Grill,Nashville,0.015967
5,Benchmark Bar & Grill,Nashville,0.035722
6,Famous Nashville,Nashville,0.035978
7,Pita Pit,Nashville,0.043481
8,Nashville Street Tacos,Nashville,0.059116
9,Hard Rock Cafe Sales,Nashville,0.066007


In [11]:
taxonomy_keep = taxonomy[taxonomy["Keep"] == "Yes"]

In [12]:
mapping = taxonomy_keep.set_index("Category").to_dict("index")

In [13]:
categories = [
    c.strip()
    for c in restaurants.iloc[0]["categories"].split(",")
]

In [14]:
for c in categories:
    if c in mapping:
        print(c, "->", mapping[c])

Sushi Bars -> {'Count': 1717, 'Feature Type': 'Restaurant Type', 'Standardized Value': 'Sushi', 'Keep': 'Yes', 'Notes': nan}
Japanese -> {'Count': 1830, 'Feature Type': 'Cuisine', 'Standardized Value': 'Japanese', 'Keep': 'Yes', 'Notes': nan}


In [15]:
def extract_features(category_string, mapping):

    features = {
        "Cuisine": set(),
        "Restaurant Type": set(),
        "Experience": set()
    }

    categories = [
        category.strip()
        for category in category_string.split(",")
    ]

    for category in categories:

        if category in mapping:

            info = mapping[category]

            feature_type = info["Feature Type"]

            value = info["Standardized Value"]

            if feature_type in features and pd.notna(value):
                features[feature_type].add(value)

    return {
        key: list(value)
        for key, value in features.items()
    }

In [16]:
restaurant = restaurants.iloc[0]["categories"]

extract_features(restaurant,mapping)



{'Cuisine': ['Japanese'], 'Restaurant Type': ['Sushi'], 'Experience': []}

In [17]:
restaurants["Extracted Features"] = restaurants["categories"].apply(
    lambda x: extract_features(x, mapping)
)

In [18]:
restaurants[["name", "Extracted Features"]].head()

,name,Extracted Features
0,Ichiban,"{'Cuisine': ['Japanese'], 'Restaurant Type': [..."
1,REPUBLIC quality food & drink,"{'Cuisine': [], 'Restaurant Type': ['Gastropub..."
2,Music City Chicken,"{'Cuisine': ['American', 'Southern'], 'Restaur..."
3,The Stillery,"{'Cuisine': ['American'], 'Restaurant Type': [..."
4,Piranha's Bar & Grill,"{'Cuisine': ['American'], 'Restaurant Type': [..."


In [19]:
restaurants["Restaurant Type"] = restaurants["Extracted Features"].apply(
    lambda features: features["Restaurant Type"]
)

restaurants["Experience"] = restaurants["Extracted Features"].apply(
    lambda features: features["Experience"]
)

restaurants["Cuisine"] = restaurants["Extracted Features"].apply(
    lambda features: features["Cuisine"]
)

In [20]:
from sklearn.preprocessing import MultiLabelBinarizer

mlb = MultiLabelBinarizer()

cuisine_mlb = mlb.fit_transform(restaurants["Cuisine"])

cuisine_mlb

array([[0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 1, 0, ..., 0, 0, 0],
       ...,
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0],
       [0, 0, 0, ..., 0, 0, 0]], shape=(2034, 60))

In [21]:
Cuisine = mlb.inverse_transform(cuisine_mlb)

Cuisine

[('Japanese',),
 (),
 ('American', 'Southern'),
 ('American',),
 ('American',),
 ('American',),
 ('American', 'Southern'),
 ('Greek', 'Mediterranean'),
 ('Mexican',),
 (),
 (),
 ('American',),
 (),
 ('Mexican',),
 (),
 ('American', 'Southern'),
 ('American', 'Southern'),
 ('American',),
 ('American',),
 ('Mexican',),
 ('American',),
 ('Mediterranean', 'Middle Eastern'),
 ('American',),
 ('American',),
 (),
 ('American',),
 ('Cajun/Creole', 'Tex-Mex'),
 ('Mexican',),
 ('Mexican',),
 (),
 ('American',),
 ('American',),
 ('Southern',),
 ('Italian',),
 ('American',),
 ('American',),
 (),
 ('American', 'Southern'),
 (),
 (),
 ('Persian/Iranian',),
 ('Mexican',),
 ('American',),
 ('Korean',),
 (),
 ('American',),
 (),
 (),
 ('American', 'Southern'),
 ('American',),
 ('American',),
 ('Southern',),
 ('Greek', 'Mediterranean', 'Middle Eastern'),
 ('Italian',),
 ('American',),
 ('Asian Fusion',),
 ('Mediterranean',),
 ('American', 'Italian'),
 ('American',),
 ('American',),
 ('American',),
 ('Am

In [22]:
cuisine_df = pd.DataFrame(cuisine_mlb,columns=mlb.classes_)

In [23]:
restaurants["Restaurant Type"].apply(type).value_counts()

Restaurant Type
<class 'list'>    2034
Name: count, dtype: int64

In [24]:
all_types = set()

for lst in restaurants["Restaurant Type"]:
    all_types.update(type(x) for x in lst)

all_types

{str}

In [25]:
restaurants[
    restaurants["Restaurant Type"].apply(
        lambda lst: any(not isinstance(x, str) for x in lst)
    )
][["name", "Restaurant Type"]]

,name,Restaurant Type


In [26]:
restaurant_type_mlb = mlb.fit_transform(restaurants["Restaurant Type"])

restaurant_type = mlb.inverse_transform(restaurant_type_mlb)

restaurant_type_df = pd.DataFrame(restaurant_type_mlb, columns=mlb.classes_)

restaurant_type_df

,Acai Bowls,Alcoholic Beverages,Bagels,Bakery,Barbeque,Beer,Breakfast & Brunch,Bubble Tea,Buffets,Burgers,...,Tacos,Tapas/Small Plates,Tea,Teppanyaki,Vegan,Vegetarian,Waffles,Whiskey,Wine & Spirits,Wraps
0,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,0,0,0,0,0,1,1,0,0,0,...,0,0,0,0,0,0,0,0,1,0
3,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,1,0
4,0,0,0,0,0,1,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2029,0,0,0,0,0,0,1,0,0,0,...,1,0,0,0,0,0,0,0,0,0
2030,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2031,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2032,0,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [27]:
experience_mlb = mlb.fit_transform(restaurants["Experience"])

experience = mlb.inverse_transform(experience_mlb)

experience_df = pd.DataFrame(experience_mlb, columns=mlb.classes_)

experience_df

,Beer,Lounges,Wine
0,0,0,0
1,0,0,0
2,0,0,0
3,0,0,0
4,0,0,0
...,...,...,...
2029,0,0,0
2030,0,0,0
2031,0,0,0
2032,0,0,0


In [28]:
numeric_features_df = restaurants[
    [
        "latitude",
        "longitude",
        "stars",
        "review_count"
    ]
]

In [29]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

scaled_numeric = scaler.fit_transform(numeric_features_df[["stars","review_count"]])

scaled_numeric

scaled_numeric_df = pd.DataFrame(scaled_numeric, columns = ['stars', 'review_count'])

scaled_numeric_df


,stars,review_count
0,0.460541,0.003637
1,0.460541,-0.419133
2,1.067238,-0.035997
3,1.067238,8.075916
4,0.460541,-0.224262
...,...,...
2029,-2.572943,-0.376196
2030,1.067238,-0.422436
2031,0.460541,0.188600
2032,-2.572943,-0.412528


In [30]:
weighted_cuisine = cuisine_df * CUISINE_WEIGHT

weighted_type = restaurant_type_df * TYPE_WEIGHT

weighted_experience = experience_df * EXPERIENCE_WEIGHT

weighted_numeric = scaled_numeric_df * NUMERIC_WEIGHT

In [31]:
recommendation_feature_matrix = pd.concat(
    [
        weighted_cuisine,
        weighted_type,
        weighted_experience,
        weighted_numeric
    ],
    axis=1
)

In [32]:
def find_restaurant(
    restaurant_name,
    city,
    restaurants
):

    selected_index = restaurants[
        (restaurants["name"] == restaurant_name) &
        (restaurants["city"] == city)
    ].index[0]

    return selected_index

In [33]:
def get_feature_vector(
    selected_index,
    recommendation_feature_matrix
):

    selected_vector = recommendation_feature_matrix.loc[
        [selected_index]
    ]

    return selected_vector

In [34]:

def calculate_similarity(
    selected_vector,
    recommendation_feature_matrix
):

    similarity_scores = cosine_similarity(
        selected_vector,
        recommendation_feature_matrix
    )

    similarity_df = pd.DataFrame(
        similarity_scores.T,
        columns=["similarity"]
    )

    return similarity_df

In [35]:
def get_top_recommendations(
    similarity_df,
    restaurants,
    selected_index,
    top_n=10,
    exclude_same_chain=False
):

    filtered_df = similarity_df.drop(selected_index)

    filtered_df = filtered_df.copy()

    filtered_df["proximity_score"] = restaurants.loc[
        filtered_df.index,
        "proximity_score"
    ]

    if exclude_same_chain:
        selected_name = restaurants.loc[selected_index, "name"]

        filtered_df = filtered_df[
            restaurants.loc[filtered_df.index, "name"] != selected_name
        ]

    filtered_df["hybrid_score"] = (
        SIMILARITY_WEIGHT * filtered_df["similarity"]
        + PROXIMITY_WEIGHT * filtered_df["proximity_score"]
    )

    top_similarity_df = filtered_df.nlargest(
        top_n,
        "hybrid_score"
    )

    return top_similarity_df

In [36]:
def format_recommendations(
    top_similarity_df,
    restaurants
):

    recommendation_table = restaurants.loc[
        top_similarity_df.index
    ]

    recommendation_table = recommendation_table[
        [
            "name",
            "city",
            "state",
            "stars",
            "review_count",
            "distance_km"
        ]
    ]

    recommendation_table = pd.concat(
        [recommendation_table, top_similarity_df],
        axis=1
    )

    return recommendation_table

In [37]:
def recommend_restaurants(
    restaurant_name,
    city,
    restaurants,
    recommendation_feature_matrix,
    top_n=10
):

    selected_index = find_restaurant(
        restaurant_name,
        city,
        restaurants
    )

    selected_vector = get_feature_vector(
        selected_index,
        recommendation_feature_matrix
    )

    similarity_df = calculate_similarity(
        selected_vector,
        recommendation_feature_matrix
    )

    top_similarity_df = get_top_recommendations(
    similarity_df=similarity_df,
    restaurants=restaurants,
    selected_index=selected_index,
    top_n=top_n,
    exclude_same_chain=True
    )   

    recommendation_table = format_recommendations(
        top_similarity_df,
        restaurants
    )

    return recommendation_table

In [38]:
recommend_restaurants(
    restaurant_name="Ichiban",
    city="Nashville",
    restaurants=restaurants,
    recommendation_feature_matrix=recommendation_feature_matrix,
    top_n=10
)

,name,city,state,stars,review_count,distance_km,similarity,proximity_score,hybrid_score
110,Sam's Sushi Bar,Nashville,TN,3.5,90,0.327974,0.984792,0.967735,0.979675
229,Koto Sushi Bar,Nashville,TN,3.5,109,0.516269,0.985489,0.949683,0.974747
679,Sushi Circle,Nashville,TN,4.5,72,2.319311,0.985138,0.793001,0.927497
825,O'Sake,Nashville,TN,4.0,159,2.821511,0.999907,0.754160,0.926183
727,Ken's Sushi Japanese Restaurant,Nashville,TN,3.5,73,2.560060,0.983919,0.774137,0.920984
866,I Love Sushi Nashville,Nashville,TN,4.0,58,3.015050,0.996960,0.739704,0.919783
809,GoGo Sushi,Nashville,TN,3.5,82,2.768067,0.984411,0.758201,0.916548
821,Samurai Sushi,Nashville,TN,4.0,367,2.809278,0.980089,0.755083,0.912587
325,Wild Wasabi,Nashville,TN,4.0,331,0.991959,0.866617,0.905565,0.878302
1273,Tenno Japanese Restaurant,Nashville,TN,3.5,29,4.779248,0.980576,0.620069,0.872424


### Verify proximity score calculation

In [39]:
restaurants[
    [
        "name",
        "distance_km",
        "proximity_score"
    ]
].head(15)

,name,distance_km,proximity_score
0,Ichiban,0.000000,1.000000
1,REPUBLIC quality food & drink,0.005356,0.999465
2,Music City Chicken,0.005356,0.999465
3,The Stillery,0.015862,0.998415
4,Piranha's Bar & Grill,0.015967,0.998405
5,Benchmark Bar & Grill,0.035722,0.996434
6,Famous Nashville,0.035978,0.996409
7,Pita Pit,0.043481,0.995661
8,Nashville Street Tacos,0.059116,0.994106
9,Hard Rock Cafe Sales,0.066007,0.993421
